# GUNI Academic Portal - PYQ Intelligent Mapping Model

This notebook contains the code to train a Machine Learning model that automatically maps exam questions to their respective university subjects based on text analysis.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
import joblib
import os
import re

## 1. Load Datasets
We load the subjects list and the questions bank provided by the university.

In [ ]:
# Paths to datasets
subjects_path = '../PYQ_ALL_COURSES_DATA/Ganpat_University_All_Courses.xlsx'
questions_path = '../PYQ_ALL_COURSES_DATA/Ganpat_University_Questions.xlsx'

df_subjects = pd.read_excel(subjects_path)
df_questions = pd.read_excel(questions_path)

print(f"Loaded {len(df_subjects)} subjects.")
print(f"Loaded {len(df_questions)} questions.")

## 2. Intelligent Auto-Labeling
Since the questions list doesn't have subject codes, we map them by matching keywords from the question text against subject names within the same Course and Semester.

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

training_data = []

for index, row in df_questions.iterrows():
    q_text = str(row['Question Text'])
    q_course = row['Course']
    q_sem = row['Semester']
    
    # Find subjects for this course and semester
    mask = (df_subjects['Course'] == q_course) & (df_subjects['Semester'] == q_sem)
    possible_subjects = df_subjects[mask]
    
    best_subject = None
    max_matches = 0
    
    for _, sub_row in possible_subjects.iterrows():
        sub_name = str(sub_row['Subject Name'])
        # Simple heuristic: count occurrences of subject name keywords in question
        keywords = [k for k in sub_name.split() if len(k) > 3]
        matches = sum(1 for k in keywords if k.lower() in q_text.lower())
        
        if matches > max_matches:
            max_matches = matches
            best_subject = sub_row['Subject Code']
            
    if best_subject:
        training_data.append({
            'text': q_text,
            'label': best_subject
        })

df_train = pd.DataFrame(training_data)
print(f"Successfully labeled {len(df_train)} samples for training.")

## 3. Model Training
Using TF-IDF for vectorization and Linear Support Vector Classifier for prediction.

In [ ]:
if not df_train.empty:
    model = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english')),
        ('clf', LinearSVC(dual=False))
    ])

    model.fit(df_train['text'], df_train['label'])
    print("Model training completed.")
else:
    print("Not enough data to train model.")

## 4. Save the Model
Exporting the model to a .pkl file for use in the main Django application.

In [ ]:
output_file = 'pyq_intelligent_model.pkl'
joblib.dump(model, output_file)
print(f"Model saved to {output_file}")